# 🤖 PHASE 4: MODEL TRAINING
## Train 4 ML Models for Anomaly Classification

**Objective:** Train and tune machine learning models

**Input:** ML features (X, y)  
**Output:** Trained models + cross-validation results

**Models:**
- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import json
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except:
    XGBOOST_AVAILABLE = False

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except:
    LIGHTGBM_AVAILABLE = False

sys.path.insert(0, '../scripts')
from config import ML_FEATURES_X_FILE, ML_FEATURES_Y_FILE, MODEL_DIR
from logger import get_logger

logger = get_logger(__name__)

# Visualization
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Imports successful!")
print(f"  XGBoost available: {XGBOOST_AVAILABLE}")
print(f"  LightGBM available: {LIGHTGBM_AVAILABLE}")

## 1️⃣ LOAD DATA

In [ ]:
# Load features and labels
logger.info(f"Loading ML data...")
X = pd.read_csv(ML_FEATURES_X_FILE)
y = pd.read_csv(ML_FEATURES_Y_FILE, header=0, squeeze=True)

print(f"✅ Loaded data")
print(f"  X shape: {X.shape}")
print(f"  y shape: {len(y):,}")

# Encode labels
unique_labels = y.unique()
label_encoder = {label: i for i, label in enumerate(unique_labels)}
y_encoded = y.map(label_encoder)

print(f"\n🏷️ LABEL ENCODING:")
for label, code in label_encoder.items():
    count = (y == label).sum()
    print(f"  {label:30s} → {code}  ({count:,})")

## 2️⃣ TRAIN/TEST SPLIT

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n📊 TRAIN/TEST SPLIT:")
print(f"  Training set: {len(X_train):,} samples")
print(f"  Test set: {len(X_test):,} samples")
print(f"  Ratio: {len(X_train)/(len(X_train)+len(X_test))*100:.0f}% / {len(X_test)/(len(X_train)+len(X_test))*100:.0f}%")

print(f"\n  Training distribution:")
for label, code in label_encoder.items():
    count = (y_train == code).sum()
    print(f"    {label:30s}: {count:6,}")

## 3️⃣ DATA PREPROCESSING

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to handle class imbalance
try:
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
    print(f"✅ SMOTE applied")
    print(f"  Before SMOTE: {len(X_train_scaled):,}")
    print(f"  After SMOTE: {len(X_train_balanced):,}")
except Exception as e:
    logger.warning(f"SMOTE failed: {e}. Using original data.")
    X_train_balanced = X_train_scaled
    y_train_balanced = y_train

## 4️⃣ TRAIN MODELS

In [ ]:
# Dictionary to store models
models = {}
results = {}

# Create output directory
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

# ============================================
# 1. Logistic Regression
# ============================================
print("\n🤖 TRAINING MODELS...\n")
print("1️⃣ Logistic Regression")

lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_balanced, y_train_balanced)
models['logistic_regression'] = lr

cv_scores = cross_val_score(lr, X_train_balanced, y_train_balanced, cv=5, scoring='f1_macro')
results['logistic_regression'] = {
    'cv_mean': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'train_score': lr.score(X_train_balanced, y_train_balanced),
}
print(f"  CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"  Train Score: {results['logistic_regression']['train_score']:.4f}")

# ============================================
# 2. Random Forest
# ============================================
print("\n2️⃣ Random Forest")

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
rf.fit(X_train_balanced, y_train_balanced)
models['random_forest'] = rf

cv_scores = cross_val_score(rf, X_train_balanced, y_train_balanced, cv=5, scoring='f1_macro')
results['random_forest'] = {
    'cv_mean': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'train_score': rf.score(X_train_balanced, y_train_balanced),
}
print(f"  CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"  Train Score: {results['random_forest']['train_score']:.4f}")

# ============================================
# 3. XGBoost
# ============================================
if XGBOOST_AVAILABLE:
    print("\n3️⃣ XGBoost")
    
    xg_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    xg_model.fit(X_train_balanced, y_train_balanced)
    models['xgboost'] = xg_model
    
    cv_scores = cross_val_score(xg_model, X_train_balanced, y_train_balanced, cv=5, scoring='f1_macro')
    results['xgboost'] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'train_score': xg_model.score(X_train_balanced, y_train_balanced),
    }
    print(f"  CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Train Score: {results['xgboost']['train_score']:.4f}")
else:
    print("\n3️⃣ XGBoost - ⚠️ NOT AVAILABLE")

# ============================================
# 4. LightGBM
# ============================================
if LIGHTGBM_AVAILABLE:
    print("\n4️⃣ LightGBM")
    
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    lgb_model.fit(X_train_balanced, y_train_balanced)
    models['lightgbm'] = lgb_model
    
    cv_scores = cross_val_score(lgb_model, X_train_balanced, y_train_balanced, cv=5, scoring='f1_macro')
    results['lightgbm'] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'train_score': lgb_model.score(X_train_balanced, y_train_balanced),
    }
    print(f"  CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Train Score: {results['lightgbm']['train_score']:.4f}")
else:
    print("\n4️⃣ LightGBM - ⚠️ NOT AVAILABLE")

## 5️⃣ SAVE MODELS

In [ ]:
# Save trained models
print(f"\n💾 SAVING MODELS...\n")

for name, model in models.items():
    model_path = f"{MODEL_DIR}/{name}_model.pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    logger.info(f"✅ Saved {name} to {model_path}")
    print(f"  ✅ {name}")

# Save scaler
scaler_path = f"{MODEL_DIR}/scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
logger.info(f"✅ Saved scaler to {scaler_path}")
print(f"  ✅ scaler")

# Save label encoder
encoder_path = f"{MODEL_DIR}/label_encoder.json"
with open(encoder_path, 'w') as f:
    json.dump(label_encoder, f)
logger.info(f"✅ Saved label encoder to {encoder_path}")
print(f"  ✅ label_encoder")

## 6️⃣ MODEL REGISTRY

In [ ]:
# Create model registry
registry = {
    'timestamp': datetime.now().isoformat(),
    'data_shape': {'X': X_train_balanced.shape, 'y': len(y_train_balanced)},
    'test_size': 0.2,
    'smote_applied': True,
    'models': {},
    'label_encoder': label_encoder,
}

for name, metrics in results.items():
    registry['models'][name] = metrics

# Save registry
registry_path = f"{MODEL_DIR}/model_registry.json"
with open(registry_path, 'w') as f:
    json.dump(registry, f, indent=2)

logger.info(f"✅ Saved model registry to {registry_path}")

print(f"\n📋 MODEL REGISTRY SAVED:")
print(f"  Location: {registry_path}")
print(json.dumps(registry, indent=2)[:500] + "...")

## ✅ MODEL TRAINING SUMMARY

**Summary:**
- ✅ Loaded ML features and labels
- ✅ Train/test split (80/20)
- ✅ Scaled features (StandardScaler)
- ✅ Applied SMOTE for class balance
- ✅ Trained 4 models
- ✅ Performed cross-validation
- ✅ Saved all models
- ✅ Created model registry

**Next Step:** Proceed to 07_model_evaluation.ipynb